**_3. WAP to implement a three-layer neural network using Tensor flow library (only, no keras) to classify MNIST handwritten digits dataset. Demonstrate the implementation of feed-forward and back-propagation approaches._**

In [29]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Load MNIST dataset
mnist, info = tfds.load('mnist', with_info=True, as_supervised=True)

# Parameters
learning_rate = 0.01
num_steps = 1000
batch_size = 128
display_step = 100

# Network Parameters
n_hidden_1 = 256  # 1st layer number of neurons
n_hidden_2 = 256  # 2nd layer number of neurons
n_input = 784     # MNIST data input (img shape: 28*28)
n_classes = 10    # MNIST total classes (0-9 digits)

In [30]:
# Prepare the data
def preprocess(image, label):
    image = tf.reshape(image, [n_input])
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, n_classes)
    return image, label

train_data = mnist['train'].map(preprocess).shuffle(60000).batch(batch_size)
test_data = mnist['test'].map(preprocess).batch(batch_size)

# Create an iterator
iterator = tf.compat.v1.data.make_initializable_iterator(train_data)
next_element = iterator.get_next()

In [31]:

# Disable eager execution
tf.compat.v1.disable_eager_execution()

# tf Graph input
X = tf.compat.v1.placeholder("float", [None, n_input])
Y = tf.compat.v1.placeholder("float", [None, n_classes])

# Store layers weight & bias
weights = {
    'h1': tf.Variable(tf.random.normal([n_input, n_hidden_1])),
    'h2': tf.Variable(tf.random.normal([n_hidden_1, n_hidden_2])),
    'out': tf.Variable(tf.random.normal([n_hidden_2, n_classes]))
}
biases = {
    'b1': tf.Variable(tf.random.normal([n_hidden_1])),
    'b2': tf.Variable(tf.random.normal([n_hidden_2])),
    'out': tf.Variable(tf.random.normal([n_classes]))
}

# Create model
def neural_net(x):
    # Hidden fully connected layer with 256 neurons
    layer_1 = tf.add(tf.matmul(x, weights['h1']), biases['b1'])
    # Apply ReLU activation
    layer_1 = tf.nn.relu(layer_1)

    # Hidden fully connected layer with 256 neurons
    layer_2 = tf.add(tf.matmul(layer_1, weights['h2']), biases['b2'])
    # Apply ReLU activation
    layer_2 = tf.nn.relu(layer_2)

    # Output fully connected layer with a neuron for each class
    out_layer = tf.matmul(layer_2, weights['out']) + biases['out']
    return out_layer

In [32]:
# Construct model
logits = neural_net(X)

# Define loss and optimizer
loss_op = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=Y))
optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate=learning_rate)
train_op = optimizer.minimize(loss_op)

# Evaluate model
correct_pred = tf.equal(tf.argmax(logits, 1), tf.argmax(Y, 1))
accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))

# Initialize the variables (i.e. assign their default value)
init = tf.compat.v1.global_variables_initializer()

In [33]:
# Start training
with tf.compat.v1.Session() as sess:
    # Run the initializer
    sess.run(init)
    sess.run(iterator.initializer)

    for step in range(1, num_steps + 1):
        try:
            batch_x, batch_y = sess.run(next_element)
        except tf.errors.OutOfRangeError:
            sess.run(iterator.initializer)
            batch_x, batch_y = sess.run(next_element)
        # Run optimization op (backprop)
        sess.run(train_op, feed_dict={X: batch_x, Y: batch_y})
        if step % display_step == 0 or step == 1:
            # Calculate batch loss and accuracy
            loss, acc = sess.run([loss_op, accuracy], feed_dict={X: batch_x, Y: batch_y})
            print("Step " + str(step) + ", Minibatch Loss= " + \
                  "{:.4f}".format(loss) + ", Training Accuracy= " + \
                  "{:.3f}".format(acc))

    print("Optimization Finished!")

    # Calculate accuracy for MNIST test images
    test_iterator = tf.compat.v1.data.make_initializable_iterator(test_data)
    next_test_element = test_iterator.get_next()
    sess.run(test_iterator.initializer)
    test_acc = 0
    test_count = 0
    while True:
        try:
            test_images, test_labels = sess.run(next_test_element)
            acc = sess.run(accuracy, feed_dict={X: test_images, Y: test_labels})
            test_acc += acc
            test_count += 1
        except tf.errors.OutOfRangeError:
            break
    test_acc /= test_count
    print("Testing Accuracy:", test_acc)

Step 1, Minibatch Loss= 1359.6335, Training Accuracy= 0.133
Step 100, Minibatch Loss= 38.9361, Training Accuracy= 0.852
Step 200, Minibatch Loss= 17.8708, Training Accuracy= 0.953
Step 300, Minibatch Loss= 24.1007, Training Accuracy= 0.883
Step 400, Minibatch Loss= 10.7979, Training Accuracy= 0.953
Step 500, Minibatch Loss= 11.1566, Training Accuracy= 0.922
Step 600, Minibatch Loss= 14.5080, Training Accuracy= 0.906
Step 700, Minibatch Loss= 1.1279, Training Accuracy= 0.977
Step 800, Minibatch Loss= 3.1825, Training Accuracy= 0.953
Step 900, Minibatch Loss= 10.2285, Training Accuracy= 0.953
Step 1000, Minibatch Loss= 0.9395, Training Accuracy= 0.969
Optimization Finished!
Testing Accuracy: 0.94462025
